In [ ]:
from utils import *

In [ ]:
root_directory = "/nfs/datz/olmo_models/new_processed_outputs/"

file_paths = traverse_directory(root_directory)
#main_files = [path for path in file_paths if ("subgraph" not in path and "test" not in path and "all_country_capital" in path)]

root_directory = "/nfs/datz/olmo_models/new_processed_outputs/"

all_files = [file for file in file_paths if "test" not in file]

models_data = read_all_files(all_files)

for model_name, relations in models_data.items():
    print(f"Model: {model_name}")
    for relation_name, data_entries in relations.items():
        print(f"  Relation: {relation_name}, Number of Entries: {len(data_entries)}")
        
sorted_models = sorted(
        models_data.keys(), 
        key=lambda name: int(re.search(r'step(\d+)', name).group(1)) if "main" not in name else float('inf')
    )

In [ ]:
models_data['main']

In [ ]:
model_name = "allenai/OLMo-7B-0424-hf"

model = load_models(model_name, device='cuda:7', checkpoint="main")

In [ ]:
ffn_dict = {}

theta = 0.8

for model_name in sorted_models:
    relations_data = models_data[model_name]

    print(model_name)
    g_total = 0
    general_heatmap = np.zeros(32, dtype=float)
    e_total = 0
    entity_heatmap = np.zeros(32, dtype=float)

    relation_answer_heatmaps = {}
    relation_answer_with_specific = {}

    for relation_name, entries in relations_data.items():

        relation_answer_heatmap = np.zeros(32, dtype=float)
        relation_answer_total = 0
        answer_specific_heatmaps = {}

        for idx, entry in enumerate(entries):

            # Ensure subj_token_span is not None
            if entry["subj_token_span"] is None:
                entry["subj_token_span"] = np.arange(0, len(entry["subject_tokens"])).tolist()

            # Update general heatmap
            general_heatmap += np.mean(entry['ffns_contribution_maps'], axis=0) #added [:entry['answer_token_span'][0]] because I dont need the contributions from the answer token itself
            g_total += 1

            # Update entity heatmap for subject tokens
            one_data_map = np.zeros(32, dtype=float)
            for t in entry["subj_token_span"]:
                # if t == 0:
                #     e_total -= 1
                #     continue
                one_data_map += entry['ffns_contribution_maps'][t]#[t - 1]
            entity_heatmap += one_data_map
            e_total += len(entry["subj_token_span"])

            # Update entity heatmap and relation answer heatmap for answer tokens
            one_data_map = np.zeros(32, dtype=float)
            for t in entry["answer_token_span"]:
                one_data_map += entry['ffns_contribution_maps'][t - 1]
            entity_heatmap += one_data_map
            relation_answer_heatmap += one_data_map

            e_total += len(entry["answer_token_span"])
            relation_answer_total += len(entry["answer_token_span"])

            # Store answer-specific heatmap
            answer_specific_heatmap = one_data_map / len(entry["answer_token_span"])
            answer_specific_heatmaps[idx] = answer_specific_heatmap  > theta

        relation_answer_heatmap /= relation_answer_total
        relation_answer_heatmaps[f"{relation_name}"] = relation_answer_heatmap > theta
        relation_answer_with_specific[f"{relation_name}"] = answer_specific_heatmaps

    # Normalize heatmaps
    general_heatmap /= g_total
    entity_heatmap /= e_total

    # Store heatmaps in the dictionary
    ffn_dict[model_name] = {
        'general_heatmap': general_heatmap > theta,
        'entity_heatmap': entity_heatmap > theta,
        'relation_answer_heatmaps': relation_answer_heatmaps,
        'relation_answer_with_specific': relation_answer_with_specific
    }

In [ ]:
def calculate_proper_heads(heatmaps_dict):
    """
    Calculates proper entity heads, proper relation answer heads, 
    and proper answer specific heads for each model in heatmaps_dict.

    Args:
        heatmaps_dict (dict): A dictionary where each key is a model name and 
                              the value is a dictionary with heatmaps.

    Returns:
        dict: A dictionary containing proper heads for each model.
    """
    proper_heads_dict = {}

    for model_name, heatmaps in heatmaps_dict.items():
        # Extract heatmaps
        general_heads = heatmaps['general_heatmap']
        entity_heads = heatmaps['entity_heatmap']
        relation_answer_heads = heatmaps['relation_answer_heatmaps']
        answer_specific_heads = heatmaps['relation_answer_with_specific']

        # Calculate proper heads
        proper_entity_heads = np.logical_and(entity_heads, np.logical_not(general_heads))
        
        # Initialize dictionaries to store results for relations and specific answers
        proper_relation_answer_heads = {}
        proper_answer_specific_heads = {}

        # Calculate proper relation answer heads per relation
        for relation, relation_heads in relation_answer_heatmaps.items():
            proper_relation_answer_heads[relation] = np.logical_and(
                relation_heads,
                np.logical_not(np.logical_or(entity_heads, general_heads))
            )

        # Calculate proper answer specific heads per relation and specific answer
        for relation, specific_answers in relation_answer_with_specific.items():
            proper_answer_specific_heads[relation] = {}
            for answer_id, specific_heads in specific_answers.items():
                proper_answer_specific_heads[relation][answer_id] = np.logical_and(
                    specific_heads,
                    np.logical_not(
                        np.logical_or(
                            relation_answer_heatmaps[relation],
                            np.logical_or(entity_heads, general_heads)
                        )
                    )
                )

        # Store results in the output dictionary
        proper_heads_dict[model_name] = {
            'proper_general_heads': general_heads,
            'proper_entity_heads': proper_entity_heads,
            'proper_relation_answer_heads': proper_relation_answer_heads,
            'proper_answer_specific_heads': proper_answer_specific_heads
        }

    return proper_heads_dict

In [ ]:
def calculate_consistency_proper_heads(heatmaps_dict, relation_name, fact_idx):
    iou_results = {}

    # Retrieve the 'main' model's heatmaps
    main_heatmaps = heatmaps_dict['main']

    # Iterate through each model (excluding 'main') and each heatmap type
    for model_name, model_data in heatmaps_dict.items():
        if model_name == 'main':
            continue

        # Store IoU results for this model
        model_ious = {}

        for heatmap_type in main_heatmaps.keys():
            if heatmap_type == "proper_relation_answer_heads":
                # Index by relation_name for relation_answer_heatmaps
                main_heatmap = main_heatmaps[heatmap_type][relation_name]
                model_heatmap = model_data[heatmap_type][relation_name]
            elif heatmap_type == "proper_answer_specific_heads":
                # Index by relation_name and fact_idx for relation_answer_with_specific
                main_heatmap = main_heatmaps[heatmap_type][relation_name][fact_idx]
                model_heatmap = model_data[heatmap_type][relation_name][fact_idx]
            elif heatmap_type == "proper_entity_heads":
                main_heatmap = main_heatmaps[heatmap_type]
                model_heatmap = model_data[heatmap_type]
            elif heatmap_type == "proper_general_heads":
                main_heatmap = main_heatmaps[heatmap_type]
                model_heatmap = model_data[heatmap_type]
            else:
                continue

            # Calculate IoU
            intersection = np.logical_and(main_heatmap, model_heatmap)
            union = np.logical_or(main_heatmap, model_heatmap)

            area_of_intersection = np.sum(intersection)
            area_of_union = np.sum(union)

            # Avoid division by zero
            iou = area_of_intersection / area_of_union if area_of_union > 0 else 0

            # Store IoU result and statistics
            model_ious[heatmap_type] = {
                'iou': iou,
                'intersection': area_of_intersection,
                'union': area_of_union
            }

        # Add this model's IoU results to the main dictionary
        iou_results[model_name] = model_ious

    return iou_results

In [ ]:
def plot_proper_heads_iou_multiple(iou_results, relation_name, fact_idx):
    # Prepare data
    model_names = sorted(
        iou_results.keys(),
        key=lambda name: int(re.search(r'step(\d+)', name).group(1)) if "main" not in name else float('inf')
    )
    general_ious = []
    entity_ious = []
    answer_all_ious = []
    answer_ious = []

    # Populate lists for each heatmap type
    for model_name in model_names:
        general_ious.append(iou_results[model_name]['proper_general_heads']['iou'])
        entity_ious.append(iou_results[model_name]['proper_entity_heads']['iou'])
        answer_all_ious.append(iou_results[model_name]['proper_relation_answer_heads']['iou'])
        answer_ious.append(iou_results[model_name]['proper_answer_specific_heads']['iou'])

    # Plotting
    plt.figure(figsize=(20, 6))
    plt.plot(model_names, general_ious, label="Proper General FFNs IoU", marker='o', linestyle='-')
    plt.plot(model_names, entity_ious, label="Proper Entity FFNs IoU", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_ious, label="Proper Relation Answer FFNs IoU", marker='x', linestyle='-.')
    plt.plot(model_names, answer_ious, label="Proper Answer Specific FFNs IoU", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("IoU")
    plt.title(f"Proper FFNs IoU Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    
def plot_proper_heads_counts_intersection(iou_results, relation_name, fact_idx):
    # Prepare data
    model_names = sorted(
        iou_results.keys(),
        key=lambda x: int(x.split('step')[1].split('-')[0])
    )

    general_intersections = []
    entity_intersections = []
    answer_all_intersections = []
    answer_intersections = []

    # Populate lists for each heatmap type
    for model_name in model_names:
        general_intersections.append(iou_results[model_name]['proper_general_heads']['intersection'])
        entity_intersections.append(iou_results[model_name]['proper_entity_heads']['intersection'])
        answer_all_intersections.append(iou_results[model_name]['proper_relation_answer_heads']['intersection'])
        answer_intersections.append(iou_results[model_name]['proper_answer_specific_heads']['intersection'])

    # Plotting Intersections
    plt.figure(figsize=(20, 6))
    plt.plot(model_names, general_intersections, label="Proper General FFNs Intersections", marker='o', linestyle='-')
    plt.plot(model_names, entity_intersections, label="Proper Entity FFNs Intersections", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_intersections, label="Proper Relation Answer FFNs Intersections", marker='x', linestyle='-.')
    plt.plot(model_names, answer_intersections, label="Proper Answer Specific FFNs Intersections", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("Intersection Counts")
    plt.title(f"Proper FFNs Intersections Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

def plot_proper_heads_counts_union(iou_results, relation_name, fact_idx):
    # Prepare data
    model_names = sorted(
        iou_results.keys(),
        key=lambda x: int(x.split('step')[1].split('-')[0])
    )

    general_unions = []
    entity_unions = []
    answer_all_unions = []
    answer_unions = []

    # Populate lists for each heatmap type
    for model_name in model_names:
        general_unions.append(iou_results[model_name]['proper_general_heads']['union'])
        entity_unions.append(iou_results[model_name]['proper_entity_heads']['union'])
        answer_all_unions.append(iou_results[model_name]['proper_relation_answer_heads']['union'])
        answer_unions.append(iou_results[model_name]['proper_answer_specific_heads']['union'])

    # Plotting Intersections
    plt.figure(figsize=(20, 6))
    plt.plot(model_names, general_unions, label="Proper General FFNs Unions", marker='o', linestyle='-')
    plt.plot(model_names, entity_unions, label="Proper Entity FFNs Unions", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_unions, label="Proper Relation Answer FFNs Unions", marker='x', linestyle='-.')
    plt.plot(model_names, answer_unions, label="Proper Answer Specific FFNs Unions", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("Union Counts")
    plt.title(f"Proper FFNs Unions Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
relation_name = "country_capital_city"
fact_idx = 1
sent = models_data['main'][relation_name][fact_idx]['sentence']
proper_heads = calculate_proper_heads(ffn_dict)
overlap = calculate_consistency_proper_heads(proper_heads, relation_name, fact_idx)
plot_proper_heads_iou_multiple(overlap, relation_name, sent)
plot_proper_heads_counts_intersection(overlap, relation_name, sent)
plot_proper_heads_counts_union(overlap, relation_name, sent)

In [ ]:
relation_name = "books_written"
fact_idx = 1
sent = models_data['main'][relation_name][fact_idx]['sentence']
proper_heads = calculate_proper_heads(ffn_dict)
overlap = calculate_consistency_proper_heads(proper_heads, relation_name, fact_idx)
plot_proper_heads_iou_multiple(overlap, relation_name, sent)
plot_proper_heads_counts_intersection(overlap, relation_name, sent)
plot_proper_heads_counts_union(overlap, relation_name, sent)

In [ ]:
def calculate_consistency_proper_heads_all_facts(heatmaps_dict, relation_name):
    iou_results = {}

    # Retrieve the 'main' model's heatmaps
    main_heatmaps = heatmaps_dict['main']

    for model_name, model_data in heatmaps_dict.items():
        #if model_name == 'main':
        #    continue

        # Store IoU, intersection, and union results for this model
        model_metrics = {heatmap_type: {'iou': [], 'head_count': [], 'intersection': [], 'union': []}
                         for heatmap_type in main_heatmaps.keys()}

        for heatmap_type in main_heatmaps.keys():
            if heatmap_type == "proper_relation_answer_heads":
                # Iterate over all facts (assume array structure)
                #for fact_idx in range(len(main_heatmaps[heatmap_type][relation_name])):
                main_heatmap = main_heatmaps[heatmap_type][relation_name]
                model_heatmap = model_data[heatmap_type][relation_name]

                # Calculate metrics
                intersection = np.logical_and(main_heatmap, model_heatmap)
                union = np.logical_or(main_heatmap, model_heatmap)

                area_of_intersection = np.sum(intersection)
                area_of_union = np.sum(union)

                iou = area_of_intersection / area_of_union if area_of_union > 0 else 0

                model_metrics[heatmap_type]['iou'].append(iou)
                model_metrics[heatmap_type]['head_count'].append(model_heatmap.sum())
                model_metrics[heatmap_type]['intersection'].append(area_of_intersection)
                model_metrics[heatmap_type]['union'].append(area_of_union)

            elif heatmap_type == "proper_answer_specific_heads":
                # Iterate over all facts and answers
                #for fact_idx in range(len(main_heatmaps[heatmap_type][relation_name])):
                    #for answer_id, main_specific_heatmap in enumerate(
                    #    main_heatmaps[heatmap_type][relation_name][fact_idx]
                    #~):
                main_specific_heatmap = np.logical_or.reduce(list(main_heatmaps[heatmap_type][relation_name].values())) #main_heatmaps[heatmap_type][relation_name][fact_idx]
                model_specific_heatmap = np.logical_or.reduce(list(model_data[heatmap_type][relation_name].values())) #model_data[heatmap_type][relation_name][fact_idx]#[answer_id]

                # Calculate metrics
                intersection = np.logical_and(main_specific_heatmap, model_specific_heatmap)
                union = np.logical_or(main_specific_heatmap, model_specific_heatmap)

                area_of_intersection = np.sum(intersection)
                area_of_union = np.sum(union)

                iou = area_of_intersection / area_of_union if area_of_union > 0 else 0

                model_metrics[heatmap_type]['iou'].append(iou)
                model_metrics[heatmap_type]['head_count'].append(model_specific_heatmap.sum())
                model_metrics[heatmap_type]['intersection'].append(area_of_intersection)
                model_metrics[heatmap_type]['union'].append(area_of_union)

            else:
                # Handle proper_entity_heads and proper_general_heads
                main_heatmap = main_heatmaps[heatmap_type]
                model_heatmap = model_data[heatmap_type]

                # Calculate metrics
                intersection = np.logical_and(main_heatmap, model_heatmap)
                union = np.logical_or(main_heatmap, model_heatmap)

                area_of_intersection = np.sum(intersection)
                area_of_union = np.sum(union)

                iou = area_of_intersection / area_of_union if area_of_union > 0 else 0

                model_metrics[heatmap_type]['iou'].append(iou)
                model_metrics[heatmap_type]['head_count'].append(model_heatmap.sum())
                model_metrics[heatmap_type]['intersection'].append(area_of_intersection)
                model_metrics[heatmap_type]['union'].append(area_of_union)

        # Aggregate results for each heatmap type
        aggregated_metrics = {ht: {
            'iou': np.mean(metrics['iou']),
            'head_count': np.mean(metrics['head_count']),
            'intersection': np.sum(metrics['intersection']),
            'union': np.sum(metrics['union'])
        } for ht, metrics in model_metrics.items()}

        iou_results[model_name] = aggregated_metrics

    return iou_results

def plot_aggregated_proper_heads_iou(iou_results, relation_name, output_dir):
    # Prepare data
    model_names = sorted(
        iou_results.keys(),
        key=lambda name: int(re.search(r'step(\d+)', name).group(1)) if "main" not in name else float('inf')
    )
    general_ious = []
    entity_ious = []
    answer_all_ious = []
    answer_ious = []

    general_head_count = []
    entity_head_count = []
    answer_all_head_count = []
    answer_head_counts = []

    # Populate lists for each heatmap type
    for model_name in model_names[:-1]:
        general_ious.append(iou_results[model_name]['proper_general_heads']['iou'])
        entity_ious.append(iou_results[model_name]['proper_entity_heads']['iou'])
        answer_all_ious.append(iou_results[model_name]['proper_relation_answer_heads']['iou'])
        answer_ious.append(iou_results[model_name]['proper_answer_specific_heads']['iou'])

    for model_name in model_names:
        general_head_count.append(iou_results[model_name]['proper_general_heads']['head_count'])
        entity_head_count.append(iou_results[model_name]['proper_entity_heads']['head_count'])
        answer_all_head_count.append(iou_results[model_name]['proper_relation_answer_heads']['head_count'])
        answer_head_counts.append(iou_results[model_name]['proper_answer_specific_heads']['head_count'])

    # Create directories if not exist
    iou_dir = os.path.join(output_dir, "iou")
    count_dir = os.path.join(output_dir, "count")
    os.makedirs(iou_dir, exist_ok=True)
    os.makedirs(count_dir, exist_ok=True)

    # Plot and save IoU
    plt.figure(figsize=(20, 6))
    plt.plot(model_names[:-1], general_ious, label="Proper General FFNs IoU (Aggregated)", marker='o', linestyle='-')
    plt.plot(model_names[:-1], entity_ious, label="Proper Entity FFNs IoU (Aggregated)", marker='s', linestyle='--')
    plt.plot(model_names[:-1], answer_all_ious, label="Proper Relation Answer FFNs IoU (Aggregated)", marker='x', linestyle='-.')
    plt.plot(model_names[:-1], answer_ious, label="Proper Answer Specific FFNs IoU (Aggregated)", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(model_names[:-1])), labels=model_names[:-1], rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("Aggregated IoU")
    plt.title(f"Aggregated Proper FFNs IoU Across Models for Relation: {relation_name}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    #plt.savefig(os.path.join(iou_dir, f"{relation_name}_ffn_iou.png"))
    plt.show()
    plt.close()

    # Plot and save Count
    plt.figure(figsize=(20, 6))
    plt.plot(model_names, general_head_count, label="Proper General FFNs Count (Aggregated)", marker='o', linestyle='-')
    plt.plot(model_names, entity_head_count, label="Proper Entity FFNs Count (Aggregated)", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_head_count, label="Proper Relation Answer FFNs Count (Aggregated)", marker='x', linestyle='-.')
    plt.plot(model_names, answer_head_counts, label="Proper Answer Specific FFNs Count (Aggregated)", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("Aggregated Count")
    plt.title(f"Aggregated Proper FFNs Count Across Models for Relation: {relation_name}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    #plt.savefig(os.path.join(count_dir, f"{relation_name}_ffn_count.png"))
    plt.close()


# def plot_proper_heads_counts_intersection(iou_results, relation_name):
#     # Prepare data
#     model_names = sorted(
#         iou_results.keys(),
#         key=lambda x: int(x.split('step')[1].split('-')[0])
#     )

#     general_intersections = []
#     entity_intersections = []
#     answer_all_intersections = []
#     answer_intersections = []

#     # Populate lists for each heatmap type
#     for model_name in model_names:
#         general_intersections.append(iou_results[model_name]['proper_general_heads']['intersection'])
#         entity_intersections.append(iou_results[model_name]['proper_entity_heads']['intersection'])
#         answer_all_intersections.append(iou_results[model_name]['proper_relation_answer_heads']['intersection'])
#         answer_intersections.append(iou_results[model_name]['proper_answer_specific_heads']['intersection'])

#     # Plotting Intersections
#     plt.figure(figsize=(20, 6))
#     plt.plot(model_names, general_intersections, label="Proper General Heads Intersections", marker='o', linestyle='-')
#     plt.plot(model_names, entity_intersections, label="Proper Entity Heads Intersections", marker='s', linestyle='--')
#     plt.plot(model_names, answer_all_intersections, label="Proper Relation Answer Heads Intersections", marker='x', linestyle='-.')
#     plt.plot(model_names, answer_intersections, label="Proper Answer Specific Heads Intersections", marker='^', linestyle='-.')

#     plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
#     plt.xlabel("Model Steps")
#     plt.ylabel("Intersection Counts")
#     plt.title(f"Proper Heads Intersections Across Models for Relation: {relation_name}")
#     plt.legend()
#     plt.grid(True)
#     plt.tight_layout()
#     plt.show()


# def plot_proper_heads_counts_union(iou_results, relation_name):
#     # Prepare data
#     model_names = sorted(
#         iou_results.keys(),
#         key=lambda x: int(x.split('step')[1].split('-')[0])
#     )

#     general_unions = []
#     entity_unions = []
#     answer_all_unions = []
#     answer_unions = []

#     # Populate lists for each heatmap type
#     for model_name in model_names:
#         general_unions.append(iou_results[model_name]['proper_general_heads']['union'])
#         entity_unions.append(iou_results[model_name]['proper_entity_heads']['union'])
#         answer_all_unions.append(iou_results[model_name]['proper_relation_answer_heads']['union'])
#         answer_unions.append(iou_results[model_name]['proper_answer_specific_heads']['union'])

#     # Plotting Unions
#     plt.figure(figsize=(20, 6))
#     plt.plot(model_names, general_unions, label="Proper General Heads Unions", marker='o', linestyle='-')
#     plt.plot(model_names, entity_unions, label="Proper Entity Heads Unions", marker='s', linestyle='--')
#     plt.plot(model_names, answer_all_unions, label="Proper Relation Answer Heads Unions", marker='x', linestyle='-.')
#     plt.plot(model_names, answer_unions, label="Proper Answer Specific Heads Unions", marker='^', linestyle='-.')

#     plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
#     plt.xlabel("Model Steps")
#     plt.ylabel("Union Counts")
#     plt.title(f"Proper Heads Unions Across Models for Relation: {relation_name}")
#     plt.legend()
#     plt.grid(True)
#     plt.tight_layout()
#     plt.show()

In [ ]:
relation_names = models_data['main'].keys()

In [ ]:
# Main loop for all relations
output_dir = "figures"
for relation_name in relation_names:
    # Extract the sentences for all facts (optional, for reference)
    sentences = [models_data['main'][relation_name][fact_idx]['sentence'] for fact_idx in range(len(models_data['main'][relation_name]))]

    # Step 1: Calculate proper heads
    proper_heads = calculate_proper_heads(ffn_dict)

    # Step 2: Calculate overlap (IoU) across all facts for the relation
    overlap = calculate_consistency_proper_heads_all_facts(proper_heads, relation_name)

    # Step 3: Plot aggregated IoU across all facts for the relation
    plot_aggregated_proper_heads_iou(overlap, relation_name, output_dir)


In [ ]:
def aggregate_answer_specific_heads(data):
    """
    Aggregate proper_answer_specific_heads into a single 32x32 map.
    """
    return np.logical_or.reduce(
        [np.logical_or.reduce(list(fact_maps.values())) for fact_maps in list(data.values())]
    )

def aggregate_relation_answer_heads(data):
    """
    Aggregate proper_relation_answer_heads into a single 32x32 map.
    """
    return np.logical_or.reduce(list(data.values()))

revisions = sorted_models

map_types = ["proper_general_heads", "proper_entity_heads", 
             "proper_relation_answer_heads", "proper_answer_specific_heads"]

proper_heads = calculate_proper_heads(ffn_dict)

In [ ]:
revisions_2 = ['step5000-tokens20B', 'step100000-tokens419B', 'main']

In [ ]:
results = {map_type: {} for map_type in map_types}

for map_type in map_types:
    for i in range(len(revisions_2) - 1):
        rev1, rev2 = revisions_2[i], revisions_2[i + 1]

        # Get the data for the current map_type
        data1 = proper_heads[rev1][map_type]
        data2 = proper_heads[rev2][map_type]

        # Aggregate maps if necessary
        if map_type == "proper_answer_specific_heads":
            data1_agg = aggregate_answer_specific_heads(data1)
            data2_agg = aggregate_answer_specific_heads(data2)
        elif map_type == "proper_relation_answer_heads":
            data1_agg = aggregate_relation_answer_heads(data1)
            data2_agg = aggregate_relation_answer_heads(data2)
        else:
            data1_agg = data1
            data2_agg = data2

        # Step 1: Detect true-to-false and false-to-true transitions
        true_to_false = np.logical_and(data1_agg, np.logical_not(data2_agg))
        false_to_true = np.logical_and(np.logical_not(data1_agg), data2_agg)

        # Step 2: Track role switches
        switched_into_other_roles = np.zeros_like(true_to_false, dtype=bool)
        stemming_from_other_roles = np.zeros_like(false_to_true, dtype=bool)

        role_switch_counts = {"general": 0, "entity": 0, "relation_answer": 0, "answer_specific": 0}
        role_stemming_counts = {"general": 0, "entity": 0, "relation_answer": 0, "answer_specific": 0}

        switched_indices = []
        stemmed_indices = []

        for other_type in map_types:
            if other_type != map_type:
                other_data1 = proper_heads[rev1][other_type]
                other_data2 = proper_heads[rev2][other_type]

                # Aggregate if relation-specific
                #if isinstance(other_data1, dict):
                if other_type == "proper_answer_specific_heads":
                    other_data1_agg = aggregate_answer_specific_heads(other_data1)
                    other_data2_agg = aggregate_answer_specific_heads(other_data2)
                elif other_type == "proper_relation_answer_heads":
                    other_data1_agg = aggregate_relation_answer_heads(other_data1)
                    other_data2_agg = aggregate_relation_answer_heads(other_data2)
                else:
                    # For global maps, use directly
                    other_data1_agg = other_data1
                    other_data2_agg = other_data2

                
                switched_into_other_roles = np.logical_or(
                    switched_into_other_roles,
                    np.logical_and(true_to_false, other_data2_agg)
                )
                
                stemming_from_other_roles = np.logical_or(
                        stemming_from_other_roles,
                        np.logical_and(false_to_true, other_data1_agg)
                )
                
                switched_mask = np.logical_and(true_to_false, other_data2_agg)
                stemmed_mask = np.logical_and(false_to_true, other_data1_agg)
                
                switched_indices.extend(list(zip(*np.where(switched_mask))))
                stemmed_indices.extend(list(zip(*np.where(stemmed_mask))))

                # Contribution counts
                role_switch_contribution = np.sum(switched_mask)
                role_stemming_contribution = np.sum(stemmed_mask)
                
                role_switch_contribution = np.logical_and(true_to_false, other_data2_agg)
                role_switch_contribution_stemming = np.logical_and(false_to_true, other_data1_agg)

                if other_type == "proper_general_heads":
                    role_switch_counts["general"] += np.sum(role_switch_contribution)
                    role_stemming_counts["general"] += np.sum(role_switch_contribution_stemming)
                elif other_type == "proper_entity_heads":
                    role_switch_counts["entity"] += np.sum(role_switch_contribution)
                    role_stemming_counts["entity"] += np.sum(role_switch_contribution_stemming)
                elif other_type == "proper_relation_answer_heads":
                    role_switch_counts["relation_answer"] += np.sum(role_switch_contribution)
                    role_stemming_counts["relation_answer"] += np.sum(role_switch_contribution_stemming)
                elif other_type == "proper_answer_specific_heads":
                    role_switch_counts["answer_specific"] += np.sum(role_switch_contribution)
                    role_stemming_counts["answer_specific"] += np.sum(role_switch_contribution_stemming)
                # role_switch_contribution = np.logical_and(true_to_false, other_data2_agg)
                # # Aggregate switches into categories
                # #switched = np.logical_and(np.logical_not(other_data1_agg), other_data2_agg)
                # if other_type == "proper_general_heads":
                #     role_switch_counts["general"] += np.sum(role_switch_contribution)
                # elif other_type == "proper_entity_heads":
                #     role_switch_counts["entity"] += np.sum(role_switch_contribution)
                # elif other_type == "proper_relation_answer_heads":
                #     role_switch_counts["relation_answer"] += np.sum(role_switch_contribution)
                # elif other_type == "proper_answer_specific_heads":
                #     role_switch_counts["answer_specific"] += np.sum(role_switch_contribution)

        # Step 3: Final deactivated and new heads
        deactivated = np.logical_and(true_to_false, np.logical_not(switched_into_other_roles))
        deactivated_count = np.sum(deactivated)

        new_heads = np.logical_and(false_to_true, np.logical_not(stemming_from_other_roles))
        new_heads_count = np.sum(new_heads)

        # Store results for this map type and revision pair
        results[map_type][f"{rev1} → {rev2}"] = {
            "total_heads_rev1": np.sum(data1_agg),
            "total_heads_rev2": np.sum(data2_agg),
            "true_to_false": np.sum(true_to_false),#deactivated_count,
            "deactivated_count": deactivated_count, #np.sum(true_to_false),  # Changed heads
            "new_heads": new_heads_count,
            "role_switch_counts": role_switch_counts,  # Aggregated role switches
            "role_stemming_counts": role_stemming_counts,
            "switched_indices": switched_indices,  # Indices of switched heads
            "stemmed_indices": stemmed_indices,
        }

In [ ]:
for map_type, map_results in results.items():
    print(f"Results for {map_type}:")
    for change, data in map_results.items():
        print(f"  {change}:")
        print(f"    Total Heads (Rev1): {data['total_heads_rev1']}")
        print(f"    Total Heads (Rev2): {data['total_heads_rev2']}")
        print(f"    True to False: {data['true_to_false']}")
        print(f"    Deactivated Heads: {data['deactivated_count']}")
        print(f"    New Heads: {data['new_heads']}")
        print(f"    Role Switches: {data['role_switch_counts']}")
        print(f"    Role Stemming: {data['role_stemming_counts']}")
        print(f"    Switched heads in {map_type}: {data['switched_indices']}")
        print(f"    Stemmed heads in {map_type}: {data['stemmed_indices']}")

In [ ]:
from collections import Counter

# Initialize counters for layers
switched_layer_counts = Counter()

# Iterate through results to collect switched and stemmed head layer counts
for map_type in results:
    for transition in results[map_type]:
        # Get switched and stemmed indices
        switched_heads = results[map_type][transition]["switched_indices"]

        # Count occurrences for each layer and (layer, head) pair in switched heads
        for layer in switched_heads:
            switched_layer_counts[layer] += 1
            
# Print switched head layer frequencies
print("Layer change frequencies (Switched Heads):")
for layer, count in sorted(switched_layer_counts.items()):
    print(f"Layer {layer}: {count} times")

In [ ]:
# Define the transitions to extract
transitions = []
category_keys = ["proper_general_heads", "proper_entity_heads", "proper_relation_answer_heads", "proper_answer_specific_heads"]

# Define the value keys (column names)
value_keys = ["general", "entity", "relation_answer", "answer_specific", "deactivated_count"]

for category in results.values():
    transitions =list(category.keys())
    break# Collect all unique transitions

# Convert transitions to a sorted list for consistency
#transitions = sorted(list(transitions))
# Dictionary to store the extracted matrices
extracted_matrices = {}

# Iterate over transitions and extract data
for transition in transitions:
    extracted_matrix = np.zeros((len(category_keys) + 1, len(value_keys)))

    # Populate the matrix with correct values
    for i, category in enumerate(category_keys):  # Last category is for deactivated_count
        total_counts = {key: 0 for key in value_keys}  # Default values
        if transition in results.get(category, {}):
            total_counts.update(results[category][transition].get("role_switch_counts", {}))
            total_counts["deactivated_count"] += results[category][transition].get("deactivated_count", 0)
        extracted_matrix[i] = [total_counts[v] for v in value_keys]

    # Manually populate the last row with deactivated counts
    #extracted_matrix[-1] = [0, 0, 0, 0, sum(results[cat][transition].get("deactivated_count", 0) for cat in results.keys() if transition in results[cat])]
    new_heads_counts = {key: 0 for key in value_keys}
    for i, category in enumerate(category_keys[:-1]):  # Map new heads to the respective categories
        if transition in results.get(category, {}):
            new_heads_counts[value_keys[i]] += results[category][transition].get("new_heads", 0)

    extracted_matrix[-1] = [new_heads_counts[v] for v in value_keys]  # Assign correctly

    # Convert to DataFrame and store
    df_extracted = pd.DataFrame(
        extracted_matrix,
        index=["General Heads", "Entity Heads", "Relation Answer Heads", "Answer Specific Heads", "Deactivated Count"],
        columns=value_keys
    )
    
    extracted_matrices[transition] = df_extracted

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import math

def plot_all_heatmaps(extracted_matrices):
    num_matrices = len(extracted_matrices)
    cols = 5  # Number of heatmaps per row
    rows = math.ceil(num_matrices / cols)  # Calculate required rows

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = axes.flatten()  # Flatten in case we don't get a perfect grid

    for i, (transition, df_matrix) in enumerate(extracted_matrices.items()):
        sns.heatmap(df_matrix, annot=True, fmt=".1f", cmap="coolwarm", linewidths=0.5, ax=axes[i])
        axes[i].set_title(f"{transition}")
        axes[i].set_xlabel("Categories")
        axes[i].set_ylabel("Head Types")

    # Hide any unused subplots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

plot_all_heatmaps(extracted_matrices)

# Iterate over extracted matrices and plot heatmaps
# for transition, df_matrix in extracted_matrices.items():
#   plot_heatmap(df_matrix, f"Heatmap: {transition}")